In [ ]:
import pandas as pd
from tqdm import tqdm

def generate_data_csv(output_file='data.csv', target_rows=4000000):
    """
    Memory-efficient script to generate data.csv.
    Uses 'taxid' as the Organism label since string names are absent.
    """
    chunksize = 10000
    processed_rows = 0
    
    # Load taxa.csv and filter for thermophile taxids
    taxa_df = pd.read_csv('taxa.csv')
    thermophile_taxids = set(taxa_df[(taxa_df['temperature'] >= 60) & (taxa_df['thermophile_label'] == True)]['taxid'])
    
    with open(output_file, 'w', encoding='utf-8') as f:
        # Write the exact headers expected by my_code.py
        f.write("Entry Name,Organism,Sequence\n")
        
        # Load only 'pid', 'taxid', and 'protein_seq' from the large proteins.csv
        for chunk in tqdm(pd.read_csv('proteins.csv', usecols=['pid', 'taxid', 'protein_seq'], chunksize=chunksize), desc="Processing chunks"):
            
            # Drop rows with missing essential sequence data
            chunk = chunk.dropna(subset=['pid', 'protein_seq'])
            
            # Filter for thermophiles
            chunk = chunk[chunk['taxid'].isin(thermophile_taxids)]
            
            # Filter for sequence length between 50 and 510 inclusive
            chunk = chunk[chunk['protein_seq'].str.len().between(50, 510)]
            
            # Filter for canonical amino acids (Consistency alignment with Script 2)
            chunk = chunk[chunk['protein_seq'].str.fullmatch(r'[ACDEFGHIKLMNPQRSTVWY]+') == True]
            
            # FIX: Create a clean label out of the taxid (e.g., "Taxon_285")
            chunk['Organism'] = "Taxon_" + chunk['taxid'].astype(str)
            
            # Determine how many rows to take to hit the 40k target
            rows_to_add = min(len(chunk), target_rows - processed_rows)
            
            # Format to ['Entry Name', 'Organism', 'Sequence']
            out_chunk = chunk.iloc[:rows_to_add][['pid', 'Organism', 'protein_seq']]
            
            # Append to file
            out_chunk.to_csv(f, header=False, index=False)
            
            processed_rows += rows_to_add
            if processed_rows >= target_rows:
                break
                
    print(f"Successfully generated {output_file} with {processed_rows} rows.")

if __name__ == "__main__":
    generate_data_csv()

Processing chunks: 2390it [00:44, 54.06it/s]

Successfully generated data.csv with 373618 rows.


## What learn2thermDB actually is (and why that matters for your filters)

I checked the primary sources (Komp et al., *Sci. Data* 2023 — the learn2thermDB paper — and Komp et al., *Sci. Rep.* 2025, the **NOMELT** paper, which is the *only* published work that has actually used learn2thermDB to fine-tune a protein LM for thermostability generation). A few facts from these papers directly determine what a "high quality" subset looks like:

- learn2thermDB contains 69 million protein pairs of 250 amino acids or fewer derived from 4,739 mesophilic organisms (OGT < 40 °C) and 289 thermophilic organisms (OGT > 40 °C, up to 98 °C), built by first pairing organisms by 16S evolutionary distance, then finding homologous protein pairs within each organism pair via homology search. This is your `taxa_pairs` table (`is_pair` flag) feeding into your big `meso_pid/thermo_pid` pairs table.
- The **NOMELT** paper is the closest prior art to what you're doing (fine-tuning a protein LM to generate thermostable sequences from this exact dataset). Their filtering recipe, which took them from 69M raw pairs down to a genuinely high-quality set: pairs with a >20 K difference in optimal growth temperature, thermophilic source organism ≥333 K (60 °C) and mesophilic <313 K (40 °C); alignment coverage >95% on both strands; sequence length difference ≤10%. This produced 4.6 million protein pairs, subsequently clustered with MMseqs2 to 50% identity on the mesophilic sequences (min-seq-id=0.5, cluster-reassign=1, cluster-steps=5, s=7, max-seqs=1000, c=0.95, cov-mode=0, similarity-type=2, e=1e-3, cluster-mode=1).
- Redundancy is severe: each thermophilic sequence has on average 103.6 mesophilic counterparts, and 7.1 for the inverse — this is exactly why you need a dedup/capping step before DPO, not just a similarity filter.

On the **DPO side**, I checked how ProGen2 has actually been DPO-tuned in practice (Stocco et al.'s DPO_pLM, Widatalla et al., Ferragu et al.'s g-DPO, and a hit-maturation paper that fine-tunes ProGen2 directly with HuggingFace's DPOTrainer). The consistent framing for decoder-only, non-conditional models like ProGen2 is: protein design as an unconditional sequence generation problem — a generic prefix (e.g., a start token) is the prompt, and an autoregressive PLM defines a policy over amino acid sequences given that prefix. So `chosen`/`rejected` are full sequences under a shared trivial prompt (start token), not a shared biological prefix. This also matches the actual ProGen2 fine-tuning setup used at scale: ProGen2 is used as the pretrained model, without a supervised fine-tuning stage before DPO, using HuggingFace's DPO trainer.

Also worth knowing before you scale this: the number of possible DPO training pairs grows quadratically with the number of labeled sequences, leading to prohibitive training times, so sequence-space clustering is used to prune redundant pairs while preserving training signal — directly motivating the "cap pairs per protein + cluster-based split" step below.

## The filtering pipeline I'd use, and why

| Filter | Threshold | Rationale |
|---|---|---|
| Organism-pair validity | `is_pair == True` in taxa_pairs table | Confirms 16S-based evolutionary relatedness passed learn2therm's own QC, not just a coincidental protein hit |
| ΔOGT | thermo OGT ≥ 60 °C, meso OGT ≤ 40 °C, Δ ≥ 20 °C | NOMELT's exact criterion — the only published high-quality subset of this dataset for generative modeling |
| Alignment coverage | `query_align_cov` & `subject_align_cov` > 0.95 | Ensures near-full-length homology, not a shared domain — critical for DPO where chosen/rejected must be biologically comparable "responses" |
| Length difference | ≤10% of longer sequence | Excludes truncations, fusions, domain-loss artifacts that would make one sequence trivially "different" for non-thermal reasons |
| E-value | `local_E_value` < 1e-5 | Standard confident-homology cutoff, guards against spurious BLAST/DIAMOND hits |
| % identity band | 0.30 – 0.95 (`local_gap_compressed_percent_id`) | Lower bound keeps pairs above the twilight zone (true homologs); upper bound excludes near-duplicates (near-zero DPO learning signal, risk of same-species contamination) |
| Sequence length | 30–1024 aa | ProGen2 context practicality; excludes fragments and unwieldy long chains |
| Alphabet | Canonical 20 AAs only, no `X/B/Z/J/U/O/*` | Non-standard tokens are usually assembly/annotation artifacts and destabilize DPO loss |
| Redundancy cap | Top-K (e.g. K=1–3) thermo pairs per meso protein, ranked by bit_score | Raw data has >100 meso partners per thermo protein on average — uncapped pairs blow up combinatorics and bias the loss toward a few over-studied families |
| Clustering / split | MMSeqs2 50% identity on mesophilic sequences, split by cluster | Prevents train/val/test leakage from homologous pairs sharing near-identical mesophilic sequences (same protocol as NOMELT) |

DPO formatting: `prompt` = a fixed start token (empty/BOS, matching ProGen2's own convention), `chosen` = thermophilic sequence, `rejected` = its paired mesophilic homolog.

Now let me build the pipeline script (using DuckDB, since your pairs table alone is 8.7 GB / 68M rows — pandas will choke on this).

# Meso Helo pair dataset

In [ ]:
"""
build_l2t_dpo_dataset.py
=========================
Build a high-quality filtered dataset from learn2thermDB using pandas.
Outputs a single CSV (data.csv) with all metadata and sequences.

Pipeline:
  1. Load taxa.csv -> per-taxid OGT.
  2. Load taxa_pairs.csv -> valid organism pairs (is_pair == True).
  3. Filter organism pairs by OGT thresholds.
  4. Load protein_pairs.csv and join with valid organism pairs + protein sequences.
  5. Apply alignment, length, and sequence-quality filters.
  6. Cap redundancy: keep top-K meso partners per thermo protein.
  7. Save as data.csv.
"""

import pandas as pd
import re
from pathlib import Path

# --------------------------------------------------------------------------- #
# Config
# --------------------------------------------------------------------------- #

CANONICAL_AA = set("ACDEFGHIKLMNPQRSTVWY")

CFG = dict(
    # --- input files ---
    pairs_csv="protein_pairs.csv",      # meso_pid/thermo_pid protein-pair table
    proteins_csv="proteins.csv",        # pid -> protein_seq table
    taxa_pairs_csv="taxa_pairs.csv",   # organism-level 16S pairs, has is_pair
    taxa_csv="taxa.csv",               # taxid -> temperature (OGT), taxonomy

    # --- organism-level thresholds (NOMELT recipe) ---
    thermo_ogt_min_c=60.0,
    meso_ogt_max_c=40.0,
    min_delta_ogt_c=20.0,
    require_16s_is_pair=True,

    # --- protein-pair alignment quality thresholds ---
    min_query_cov=0.95,
    min_subject_cov=0.95,
    max_e_value=1e-5,
    min_pident=0.30,
    max_pident=0.95,
    max_len_diff_frac=0.10,

    # --- sequence-level QC ---
    min_seq_len=50,
    max_seq_len=510,

    # --- redundancy control ---
    top_k_per_thermo=2,

    # --- output ---
    output_csv="dpo_data.csv",
)


def is_canonical(seq):
    """Check if sequence contains only canonical amino acids."""
    if pd.isna(seq):
        return False
    return bool(re.fullmatch(r'[ACDEFGHIKLMNPQRSTVWY]+', seq))


# --------------------------------------------------------------------------- #
# Step 1-2: organism-level valid pairs (OGT + 16S relatedness)
# --------------------------------------------------------------------------- #

def build_valid_taxid_pairs(taxa_df, taxa_pairs_df, cfg):
    """
    Returns DataFrame of (meso_taxid, thermo_taxid) pairs that pass
    OGT-delta filter and (optionally) the 16S is_pair flag.
    """
    # Rename for clarity
    taxa_df = taxa_df.rename(columns={"temperature": "ogt_c"})

    if cfg["require_16s_is_pair"]:
        taxa_pairs_df = taxa_pairs_df[taxa_pairs_df["is_pair"] == True].copy()

    # Merge meso OGT
    valid = taxa_pairs_df.merge(
        taxa_df[["taxid", "ogt_c"]].rename(columns={"taxid": "meso_taxid", "ogt_c": "meso_ogt_c"}),
        on="meso_taxid",
        how="inner"
    )
    
    # Merge thermo OGT and thermophile_label (Consistency alignment with Script 1)
    valid = valid.merge(
        taxa_df[["taxid", "ogt_c", "thermophile_label"]].rename(columns={
            "taxid": "thermo_taxid", 
            "ogt_c": "thermo_ogt_c", 
            "thermophile_label": "thermo_label"
        }),
        on="thermo_taxid",
        how="inner"
    )

    # Apply OGT thresholds and thermophile label
    valid["delta_ogt_c"] = valid["thermo_ogt_c"] - valid["meso_ogt_c"]
    valid = valid[
        (valid["meso_ogt_c"] <= cfg["meso_ogt_max_c"]) &
        (valid["thermo_ogt_c"] >= cfg["thermo_ogt_min_c"]) &
        (valid["delta_ogt_c"] >= cfg["min_delta_ogt_c"]) &
        (valid["thermo_label"] == True)
    ]

    return valid[["meso_taxid", "thermo_taxid", "meso_ogt_c", "thermo_ogt_c", "delta_ogt_c"]].drop_duplicates()


# --------------------------------------------------------------------------- #
# Step 3-4: filter protein pairs + join sequences + sequence QC
# --------------------------------------------------------------------------- #

def build_filtered_pairs(pairs_df, proteins_df, valid_taxa_df, cfg):
    # Rename proteins columns for clarity
    proteins_df = proteins_df.rename(columns={"protein_seq": "seq"})

    # Join with valid taxa pairs
    df = pairs_df.merge(
        valid_taxa_df,
        on=["meso_taxid", "thermo_taxid"],
        how="inner"
    )

    # Join mesophilic sequences
    df = df.merge(
        proteins_df[["pid", "seq"]].rename(columns={"pid": "meso_pid", "seq": "meso_seq"}),
        on="meso_pid",
        how="inner"
    )

    # Join thermophilic sequences
    df = df.merge(
        proteins_df[["pid", "seq"]].rename(columns={"pid": "thermo_pid", "seq": "thermo_seq"}),
        on="thermo_pid",
        how="inner"
    )

    # Alignment quality filters
    df = df[
        (df["query_align_cov"] >= cfg["min_query_cov"]) &
        (df["subject_align_cov"] >= cfg["min_subject_cov"]) &
        (df["local_E_value"] <= cfg["max_e_value"]) &
        (df["local_gap_compressed_percent_id"] >= cfg["min_pident"]) &
        (df["local_gap_compressed_percent_id"] <= cfg["max_pident"])
    ]

    # Sequence length filters
    df["meso_len"] = df["meso_seq"].str.len()
    df["thermo_len"] = df["thermo_seq"].str.len()
    df["len_diff_frac"] = (
        (df["meso_len"] - df["thermo_len"]).abs() /
        df[["meso_len", "thermo_len"]].max(axis=1)
    )

    df = df[
        (df["meso_len"] >= cfg["min_seq_len"]) &
        (df["meso_len"] <= cfg["max_seq_len"]) &
        (df["thermo_len"] >= cfg["min_seq_len"]) &
        (df["thermo_len"] <= cfg["max_seq_len"]) &
        (df["len_diff_frac"] <= cfg["max_len_diff_frac"])
    ]

    # Canonical amino-acid alphabet only
    df = df[df["meso_seq"].apply(is_canonical) & df["thermo_seq"].apply(is_canonical)]

    # Rename percent_id for consistency
    df = df.rename(columns={"local_gap_compressed_percent_id": "pident"})

    # Select and order final columns
    cols = [
        "meso_pid", "thermo_pid",
        "meso_taxid", "thermo_taxid",
        "meso_ogt_c", "thermo_ogt_c", "delta_ogt_c",
        "pident",
        "local_E_value",
        "query_align_cov", "subject_align_cov",
        "bit_score",
        "meso_seq", "thermo_seq",
        "meso_len", "thermo_len", "len_diff_frac",
    ]
    # Keep only columns that exist
    cols = [c for c in cols if c in df.columns]
    return df[cols].copy()


# --------------------------------------------------------------------------- #
# Step 5: redundancy control — top-K meso partners per thermo protein
# --------------------------------------------------------------------------- #

def cap_redundancy(df, top_k):
    df = df.sort_values(["thermo_pid", "delta_ogt_c", "bit_score"],
                        ascending=[True, False, False])
    df = df.groupby("thermo_pid").head(top_k).reset_index(drop=True)
    n = len(df)
    print(f"[step5] capped to top-{top_k} per thermo protein -> {n:,} pairs")
    return df


# --------------------------------------------------------------------------- #
# Main
# --------------------------------------------------------------------------- #

def main(cfg):
    print("[main] Loading taxa.csv ...")
    taxa_df = pd.read_csv(cfg["taxa_csv"])

    print("[main] Loading taxa_pairs.csv ...")
    taxa_pairs_df = pd.read_csv(cfg["taxa_pairs_csv"])

    print("[main] Building valid taxid pairs ...")
    valid_taxa_df = build_valid_taxid_pairs(taxa_df, taxa_pairs_df, cfg)
    print(f"[main] {len(valid_taxa_df):,} valid organism pairs")

    print("[main] Loading protein_pairs.csv ...")
    pairs_df = pd.read_csv(cfg["pairs_csv"])

    print("[main] Loading proteins.csv ...")
    proteins_df = pd.read_csv(cfg["proteins_csv"])

    print("[main] Filtering and joining ...")
    filtered_df = build_filtered_pairs(pairs_df, proteins_df, valid_taxa_df, cfg)
    print(f"[main] {len(filtered_df):,} pairs after quality filters")

    print("[main] Capping redundancy ...")
    final_df = cap_redundancy(filtered_df, cfg["top_k_per_thermo"])

    print(f"[main] Writing {cfg['output_csv']} ...")
    final_df.to_csv(cfg["output_csv"], index=False)
    print(f"[main] Done. Output: {cfg['output_csv']} ({len(final_df):,} rows)")


if __name__ == "__main__":
    main(CFG)

[main] Loading taxa.csv ...
[main] Loading taxa_pairs.csv ...
[main] Building valid taxid pairs ...
[main] 33,022 valid organism pairs
[main] Loading protein_pairs.csv ...
[main] Loading proteins.csv ...


/var/folders/xw/ncyd0rkd4_nbf714jz6gctl80000gn/T/ipykernel_17984/3968528444.py:215: DtypeWarning: Columns (0: pdb_id) have mixed types. Specify dtype option on import or set low_memory=False.
  proteins_df = pd.read_csv(cfg["proteins_csv"])


[main] Filtering and joining ...
[main] 3,379,880 pairs after quality filters
[main] Capping redundancy ...
[step5] capped to top-2 per thermo protein -> 65,532 pairs
[main] Writing dpo_data.csv ...
[main] Done. Output: dpo_data.csv (65,532 rows)
